<a href="https://colab.research.google.com/github/Abderrahmen-Ayedi/IOT_PROJECT/blob/main/predection_pollution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle xgboost pandas scikit-learn

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('data/air_quality_dataset.csv')

In [4]:
print(df.head())

        PM2.5        PM10         NOx        NO2        SO2        VOCs  \
0   39.967142   57.926035  116.192213  55.230299   4.531693   75.317261   
1  101.935672  150.774299   76.826826  79.051618  18.744780  145.083987   
2   70.996192  138.948796  158.731020  60.466604  14.892239  145.147338   
3   28.464728   63.643900   25.385343  15.333286   7.647429  130.022319   
4   78.265276  113.977926  105.644340  59.202337  17.696806  181.713667   

         CO         CO2       CH4  Temperature   Humidity  Wind_Direction  \
0  2.789606  427.674347  1.706105    31.085120  45.454749             276   
1  1.966569  529.739619  2.492663    33.711103  60.798212             134   
2  2.626446  499.889443  2.431165    33.778698  54.875669               1   
3  1.779360  388.283712  1.818563    31.565877  67.113319             251   
4  3.240533  464.739197  2.597225    32.229835  37.236519             326   

  Location_Type     Source_Label  
0         Urban        Vehicular  
1    Industrial 

In [5]:
df = df.dropna()

In [6]:
df_indus = df[df['Location_Type'] == 'Industrial'].copy()

print(df.shape[0], "lignes au total")
print(df_indus.shape[0], "lignes industrielles")

df = df_indus

500 lignes au total
178 lignes industrielles


In [7]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
n_clean = 60

clean_indus = pd.DataFrame({
    'PM2.5': np.random.uniform(5, 15, n_clean),
    'PM10': np.random.uniform(10, 25, n_clean),
    'NOx': np.random.uniform(1, 10, n_clean),
    'NO2': np.random.uniform(1, 8, n_clean),
    'SO2': np.random.uniform(0.5, 3, n_clean),
    'VOCs': np.random.uniform(0, 10, n_clean),
    'CO': np.random.uniform(0.1, 0.8, n_clean),
    'CO2': np.random.uniform(350, 450, n_clean),
    'CH4': np.random.uniform(0.5, 1.5, n_clean),
    'Temperature': np.random.uniform(15, 25, n_clean),
    'Humidity': np.random.uniform(40, 70, n_clean),
    'Wind_Direction': np.random.uniform(0, 360, n_clean),
    'Location_Type': ['Industrial'] * n_clean,
    'Source_Label': ['Industrial'] * n_clean
})

df_indus_aug = pd.concat([df_indus, clean_indus], ignore_index=True)
print("Lignes industrielles après augmentation :", df_indus_aug.shape[0])

Lignes industrielles après augmentation : 238


In [8]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [9]:
features= [
    'PM10',
    'CO2',
    'CH4',
    'VOCs',
    'CO',
    'Temperature',
    'Humidity',
    'Wind_Direction'
]
X = df_indus_aug[features]
y = df_indus_aug['PM2.5']

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE (industriel seulement) :", rmse)

RMSE (industriel seulement) : 14.155513655174913


In [19]:
usine_sale = pd.DataFrame({
    'PM10': [80],
    'CO2': [600],
    'CH4': [3.0],
    'VOCs': [80],
    'CO': [3.0],
    'Temperature': [32],
    'Humidity': [40],
    'Wind_Direction': [270.0]
})

pm_pred_usine_sale = model.predict(usine_sale)
print('PM2.5',pm_pred_usine_sale[0], "μg/m³")

PM2.5 77.476166 μg/m³


In [18]:
usine_propre = pd.DataFrame({
    'PM10': [15],
    'CO2': [400],
    'CH4': [0.8],
    'VOCs': [5],
    'CO': [0.3],
    'Temperature': [20],
    'Humidity': [55],
    'Wind_Direction': [180.0]
})

pm_pred_usine_propre = model.predict(usine_propre)
print(pm_pred_usine_propre[0], "μg/m³")

11.264218 μg/m³


In [15]:
import pickle

with open('xgb_industrial_model.pkl', 'wb') as f:
    pickle.dump(model, f)

In [16]:
X.head()
X.shape
X.describe()

,PM10,CO2,CH4,VOCs,CO,Temperature,Humidity,Wind_Direction
count,238.000000,238.000000,238.000000,238.000000,238.000000,238.000000,238.000000,238.000000
mean,92.924406,475.825530,2.143333,114.185428,2.335115,29.516813,50.497273,191.612138
std,46.461406,52.702356,0.734584,69.018373,1.259172,5.993678,9.120896,103.167849
min,10.016467,351.467194,0.533517,0.071058,0.104165,15.188951,35.077851,1.000000
25%,33.060930,444.167026,1.464743,22.907125,0.878605,25.873993,43.106494,101.177259
50%,110.841624,494.209933,2.407852,139.326816,2.647772,32.036009,50.358865,190.538755
75%,127.153750,510.482525,2.681430,163.940923,3.235681,33.634049,57.746119,286.048662
max,171.594187,558.924257,3.352266,231.926017,4.798935,37.371560,69.900558,359.000000
